# Logistic Regression — Solar PV Risk Score Model
### RTU Anomaly Detection Training & Evaluation

**What this notebook does:**
- Loads `training_dataset.csv` produced by `anomaly_injection.py`
- Trains Logistic Regression on telemetry + synthetic image score features
- Evaluates model performance with full diagnostics
- Validates that real vs synthetic rows perform comparably
- Exports trained model for deployment at the Fog layer

**Input features (12 total):**
```
Telemetry:  PR, TEMP_DELTA, DC_AC_RATIO, PR_ROLL_MEAN,
            PR_ROLL_STD, PR_SLOPE, PR_DEV, TEMP_DELTA_SIGMA
Image:      img_panel_score, img_dusty_score,
            img_cracked_score, img_bird_drop_score
```
**Target:** `label`  (0 = normal, 1 = anomaly)

## 0 — Install Dependencies

In [ ]:
import sys
!{sys.executable} -m pip install pandas numpy scikit-learn matplotlib seaborn joblib --quiet
print('Dependencies ready.')

## 1 — Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, precision_recall_curve,
    average_precision_score, brier_score_loss
)
from sklearn.calibration import calibration_curve
from sklearn.pipeline import Pipeline

print('All imports successful.')

## 2 — Configuration

In [ ]:
# ── UPDATE THIS PATH to your training_dataset.csv ──────────
DATASET_PATH = 'training_dataset.csv'

# Feature columns fed into logistic regression
FEATURE_COLS = [
    # Telemetry features
    'PR',
    'TEMP_DELTA',
    'DC_AC_RATIO',
    'PR_ROLL_MEAN',
    'PR_ROLL_STD',
    'PR_SLOPE',
    'PR_DEV',
    'TEMP_DELTA_SIGMA',
    # Synthetic image scores
    'img_panel_score',
    'img_dusty_score',
    'img_cracked_score',
    'img_bird_drop_score',
]

TARGET_COL   = 'label'       # 0=normal, 1=anomaly
FAULT_COL    = 'fault_type'  # for breakdown analysis
SYNTH_COL    = 'synthetic'   # real vs synthetic flag

TEST_SIZE    = 0.20
RANDOM_STATE = 42

print(f'Dataset path: {DATASET_PATH}')
print(f'Feature count: {len(FEATURE_COLS)}')

## 3 — Load & Inspect Dataset

In [ ]:
df = pd.read_csv(DATASET_PATH)

print(f'Total rows:    {len(df):,}')
print(f'Total columns: {df.shape[1]}')
print(f'\n=== CLASS DISTRIBUTION ===')
label_counts = df[TARGET_COL].value_counts()
for label, count in label_counts.items():
    name = 'normal' if label == 0 else 'anomaly'
    print(f'  {name} ({label}): {count:,}  ({count/len(df)*100:.1f}%)')

print(f'\n=== FAULT TYPE BREAKDOWN ===')
for ft, count in df[FAULT_COL].value_counts().items():
    print(f'  {ft:12s}: {count:,}  ({count/len(df)*100:.1f}%)')

print(f'\n=== REAL vs SYNTHETIC ===')
for synth, count in df[SYNTH_COL].value_counts().items():
    label = 'synthetic' if synth else 'real'
    print(f'  {label:12s}: {count:,}  ({count/len(df)*100:.1f}%)')

print(f'\n=== NULL CHECK ===')
nulls = df[FEATURE_COLS + [TARGET_COL]].isnull().sum()
if nulls.sum() == 0:
    print('  No nulls found.')
else:
    print(nulls[nulls > 0])

## 4 — Feature Distribution Analysis

In [ ]:
print('=== FEATURE STATISTICS BY CLASS ===')
stats = df.groupby(TARGET_COL)[FEATURE_COLS].mean().round(4)
stats.index = ['normal', 'anomaly']
print(stats.T.to_string())

print('\n=== FEATURE CORRELATION WITH TARGET ===')
corr = df[FEATURE_COLS + [TARGET_COL]].corr()[TARGET_COL].drop(TARGET_COL)
corr_sorted = corr.abs().sort_values(ascending=False)
for feat, val in corr_sorted.items():
    direction = '+' if corr[feat] > 0 else '-'
    bar = '█' * int(abs(val) * 20)
    print(f'  {feat:25s} {direction}{val:.4f}  {bar}')

## 5 — Train / Test Split

In [ ]:
X = df[FEATURE_COLS].values
y = df[TARGET_COL].values
synthetic_flag = df[SYNTH_COL].values
fault_type     = df[FAULT_COL].values

X_train, X_test, y_train, y_test, synth_train, synth_test, fault_train, fault_test = \
    train_test_split(
        X, y, synthetic_flag, fault_type,
        test_size    = TEST_SIZE,
        random_state = RANDOM_STATE,
        stratify     = y
    )

print(f'Train set: {len(X_train):,} rows')
print(f'Test set:  {len(X_test):,} rows')
print(f'\nTrain class balance:')
for label, count in zip(*np.unique(y_train, return_counts=True)):
    print(f'  {label}: {count:,}  ({count/len(y_train)*100:.1f}%)')
print(f'\nTest class balance:')
for label, count in zip(*np.unique(y_test, return_counts=True)):
    print(f'  {label}: {count:,}  ({count/len(y_test)*100:.1f}%)')

## 6 — Train Logistic Regression

In [ ]:
# Pipeline: StandardScaler → LogisticRegression
# StandardScaler is critical — features are on very different scales
# (PR is ~1300, img scores are 0-1)
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  LogisticRegression(
        C             = 1.0,
        max_iter      = 1000,
        random_state  = RANDOM_STATE,
        class_weight  = 'balanced',  # handles residual imbalance
        solver        = 'lbfgs',
    ))
])

pipeline.fit(X_train, y_train)
print('Model trained.')

# Predictions
y_pred       = pipeline.predict(X_test)
y_prob       = pipeline.predict_proba(X_test)[:, 1]  # P(anomaly)

print(f'\n=== CLASSIFICATION REPORT ===')
print(classification_report(y_test, y_pred, target_names=['normal','anomaly']))

## 7 — Core Evaluation Metrics

In [ ]:
roc_auc = roc_auc_score(y_test, y_prob)
avg_prec = average_precision_score(y_test, y_prob)
brier    = brier_score_loss(y_test, y_prob)

print('=== CORE METRICS ===')
print(f'  ROC-AUC:           {roc_auc:.4f}   (1.0 = perfect)')
print(f'  Avg Precision:     {avg_prec:.4f}   (area under PR curve)')
print(f'  Brier Score:       {brier:.4f}   (lower is better, 0 = perfect)')

print('\n=== INTERPRETATION ===')
if roc_auc >= 0.90:
    print('  ROC-AUC: EXCELLENT — model strongly separates classes')
elif roc_auc >= 0.80:
    print('  ROC-AUC: GOOD — acceptable for production')
elif roc_auc >= 0.70:
    print('  ROC-AUC: FAIR — review feature engineering')
else:
    print('  ROC-AUC: POOR — heuristics or features need rework')

if brier <= 0.10:
    print('  Brier:   WELL CALIBRATED — risk scores are trustworthy')
elif brier <= 0.20:
    print('  Brier:   ACCEPTABLE calibration')
else:
    print('  Brier:   POORLY CALIBRATED — risk scores need calibration layer')

## 8 — Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

print('=== CONFUSION MATRIX ===')
print(f'                Predicted Normal  Predicted Anomaly')
print(f'  Actual Normal      {tn:6d}            {fp:6d}')
print(f'  Actual Anomaly     {fn:6d}            {tp:6d}')

print(f'\n=== DERIVED METRICS ===')
sensitivity = tp / (tp + fn) if (tp+fn) > 0 else 0
specificity = tn / (tn + fp) if (tn+fp) > 0 else 0
ppv         = tp / (tp + fp) if (tp+fp) > 0 else 0
npv         = tn / (tn + fn) if (tn+fn) > 0 else 0
print(f'  Sensitivity (Recall):  {sensitivity:.4f}  — of real anomalies, how many caught')
print(f'  Specificity:           {specificity:.4f}  — of real normals, how many correctly cleared')
print(f'  Precision (PPV):       {ppv:.4f}  — of flagged anomalies, how many real')
print(f'  NPV:                   {npv:.4f}  — of cleared panels, how many truly normal')

print(f'\n  False Positive Rate:   {fp/(fp+tn):.4f}  — unnecessary maintenance dispatches')
print(f'  False Negative Rate:   {fn/(fn+tp):.4f}  — missed real anomalies (the dangerous one)')

## 9 — ROC & Precision-Recall Curves

In [ ]:
fpr, tpr, roc_thresh = roc_curve(y_test, y_prob)
prec, rec, pr_thresh = precision_recall_curve(y_test, y_prob)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# ROC
axes[0].plot(fpr, tpr, 'b-', lw=2, label=f'ROC AUC = {roc_auc:.3f}')
axes[0].plot([0,1],[0,1],'k--', lw=1, label='Random')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# PR curve
axes[1].plot(rec, prec, 'r-', lw=2, label=f'AP = {avg_prec:.3f}')
axes[1].axhline(y=y_test.mean(), color='k', linestyle='--', lw=1, label='Baseline')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('roc_pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: roc_pr_curves.png')

## 10 — Threshold Analysis

In [ ]:
# Find optimal threshold — maximise F1
thresholds = np.arange(0.1, 0.9, 0.01)
results_thresh = []

for t in thresholds:
    preds = (y_prob >= t).astype(int)
    tp_ = ((preds == 1) & (y_test == 1)).sum()
    fp_ = ((preds == 1) & (y_test == 0)).sum()
    fn_ = ((preds == 0) & (y_test == 1)).sum()
    tn_ = ((preds == 0) & (y_test == 0)).sum()
    prec_ = tp_ / (tp_ + fp_) if (tp_+fp_) > 0 else 0
    rec_  = tp_ / (tp_ + fn_) if (tp_+fn_) > 0 else 0
    f1_   = 2*prec_*rec_ / (prec_+rec_) if (prec_+rec_) > 0 else 0
    fpr_  = fp_ / (fp_ + tn_) if (fp_+tn_) > 0 else 0
    results_thresh.append({'threshold': t, 'precision': prec_,
                           'recall': rec_, 'f1': f1_, 'fpr': fpr_})

thresh_df   = pd.DataFrame(results_thresh)
best_thresh = thresh_df.loc[thresh_df['f1'].idxmax()]

print('=== OPTIMAL THRESHOLD (max F1) ===')
print(f'  Threshold:  {best_thresh.threshold:.2f}')
print(f'  Precision:  {best_thresh.precision:.4f}')
print(f'  Recall:     {best_thresh.recall:.4f}')
print(f'  F1 Score:   {best_thresh.f1:.4f}')
print(f'  FPR:        {best_thresh.fpr:.4f}')

# RTU deployment: lower threshold = more sensitive (catch more anomalies)
# Higher threshold = more specific (fewer false alarms)
conservative = thresh_df[thresh_df['recall'] >= 0.90].iloc[-1]
print(f'\n=== HIGH SENSITIVITY THRESHOLD (recall >= 0.90) ===')
print(f'  Threshold:  {conservative.threshold:.2f}')
print(f'  Precision:  {conservative.precision:.4f}')
print(f'  Recall:     {conservative.recall:.4f}')
print(f'  FPR:        {conservative.fpr:.4f}')
print('  Use this for RTU — better to flag than miss a fault')

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(thresh_df.threshold, thresh_df.precision, 'b-', label='Precision')
ax.plot(thresh_df.threshold, thresh_df.recall,    'r-', label='Recall')
ax.plot(thresh_df.threshold, thresh_df.f1,        'g-', lw=2, label='F1')
ax.axvline(x=best_thresh.threshold, color='g', linestyle='--', alpha=0.7,
           label=f'Best F1 @ {best_thresh.threshold:.2f}')
ax.axvline(x=conservative.threshold, color='r', linestyle='--', alpha=0.7,
           label=f'High recall @ {conservative.threshold:.2f}')
ax.set_xlabel('Decision Threshold')
ax.set_ylabel('Score')
ax.set_title('Threshold vs Precision / Recall / F1')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('threshold_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: threshold_analysis.png')

## 11 — Per Fault Class Performance

In [ ]:
print('=== RISK SCORE DISTRIBUTION BY FAULT TYPE ===')
fault_results = []
for ft in ['normal', 'dusty', 'bird_drop', 'cracked']:
    mask  = fault_test == ft
    if mask.sum() == 0:
        continue
    scores = y_prob[mask]
    labels = y_test[mask]
    mean_score = scores.mean()
    detection_rate = (scores >= 0.5).mean() if ft != 'normal' else (scores < 0.5).mean()
    fault_results.append({
        'fault_type':     ft,
        'count':          mask.sum(),
        'mean_risk':      mean_score,
        'std_risk':       scores.std(),
        'min_risk':       scores.min(),
        'max_risk':       scores.max(),
        'correct_rate':   detection_rate
    })
    direction = 'correctly cleared' if ft == 'normal' else 'correctly flagged'
    print(f'\n  {ft.upper()}')
    print(f'    Count:          {mask.sum()}')
    print(f'    Mean risk score: {mean_score:.4f}')
    print(f'    Std:             {scores.std():.4f}')
    print(f'    {direction}: {detection_rate*100:.1f}%')

## 12 — Real vs Synthetic Row Performance

In [ ]:
# Critical check — does the model generalise beyond synthetic data?
# If real and synthetic performance differ significantly, 
# synthetic injection parameters need tightening

real_mask  = ~synth_test.astype(bool)
synth_mask =  synth_test.astype(bool)

print('=== REAL ROWS ===')
if real_mask.sum() > 0:
    real_auc = roc_auc_score(y_test[real_mask], y_prob[real_mask]) \
               if len(np.unique(y_test[real_mask])) > 1 else None
    print(f'  Count:   {real_mask.sum()}')
    print(f'  ROC-AUC: {real_auc:.4f}' if real_auc else '  ROC-AUC: N/A (single class)')
    if (y_test[real_mask] == 0).sum() > 0:
        print(f'  Mean risk (normal rows):  {y_prob[real_mask & (y_test==0)].mean():.4f}')
    if (y_test[real_mask] == 1).sum() > 0:
        print(f'  Mean risk (anomaly rows): {y_prob[real_mask & (y_test==1)].mean():.4f}')

print('\n=== SYNTHETIC ROWS ===')
if synth_mask.sum() > 0:
    synth_auc = roc_auc_score(y_test[synth_mask], y_prob[synth_mask]) \
                if len(np.unique(y_test[synth_mask])) > 1 else None
    print(f'  Count:   {synth_mask.sum()}')
    print(f'  ROC-AUC: {synth_auc:.4f}' if synth_auc else '  ROC-AUC: N/A (single class)')
    print(f'  Mean risk (anomaly rows): {y_prob[synth_mask & (y_test==1)].mean():.4f}')

print('\n=== VERDICT ===')
if real_auc and synth_auc:
    diff = abs(real_auc - synth_auc)
    if diff < 0.05:
        print(f'  AUC gap: {diff:.4f} — GOOD. Synthetic data generalises well.')
    elif diff < 0.10:
        print(f'  AUC gap: {diff:.4f} — MODERATE. Monitor on real deployment data.')
    else:
        print(f'  AUC gap: {diff:.4f} — HIGH. Synthetic injection parameters need review.')

## 13 — Cross-Validation Stability Check

In [ ]:
print('Running 5-fold stratified cross-validation...')
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

cv_auc    = cross_val_score(pipeline, X, y, cv=cv, scoring='roc_auc',    n_jobs=-1)
cv_f1     = cross_val_score(pipeline, X, y, cv=cv, scoring='f1',         n_jobs=-1)
cv_recall = cross_val_score(pipeline, X, y, cv=cv, scoring='recall',     n_jobs=-1)

print('\n=== CROSS-VALIDATION RESULTS ===')
for name, scores in [('ROC-AUC', cv_auc), ('F1', cv_f1), ('Recall', cv_recall)]:
    print(f'  {name:10s}: {scores.mean():.4f} ± {scores.std():.4f}'
          f'  (folds: {[f"{s:.3f}" for s in scores]})')

print('\n=== STABILITY VERDICT ===')
if cv_auc.std() < 0.02:
    print('  Very stable — model consistent across folds')
elif cv_auc.std() < 0.05:
    print('  Acceptable variance — minor fold-to-fold variation')
else:
    print('  High variance — model may be overfitting or data has ordering bias')

## 14 — Feature Importance

In [ ]:
scaler    = pipeline.named_steps['scaler']
logreg    = pipeline.named_steps['model']
coefs     = logreg.coef_[0]
abs_coefs = np.abs(coefs)

feat_imp = pd.DataFrame({
    'feature':    FEATURE_COLS,
    'coefficient': coefs,
    'abs_coef':   abs_coefs,
}).sort_values('abs_coef', ascending=False)

print('=== FEATURE IMPORTANCE (by absolute coefficient) ===')
print('Positive = pushes toward anomaly  |  Negative = pushes toward normal\n')
for _, row in feat_imp.iterrows():
    direction = '+' if row.coefficient > 0 else '-'
    bar = '█' * int(row.abs_coef / feat_imp.abs_coef.max() * 25)
    print(f'  {row.feature:25s} {direction}{row.abs_coef:.4f}  {bar}')

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
colors = ['tomato' if c > 0 else 'steelblue' for c in feat_imp.coefficient]
ax.barh(feat_imp.feature, feat_imp.coefficient, color=colors)
ax.axvline(x=0, color='black', lw=0.8)
ax.set_xlabel('Coefficient (log-odds)')
ax.set_title('Logistic Regression Feature Coefficients\n'
             'Red = anomaly direction | Blue = normal direction')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: feature_importance.png')

## 15 — Probability Calibration Check

In [ ]:
# Logistic Regression is generally well-calibrated
# but verify since we trained on synthetic + real data mix
# A well-calibrated model: predicted P=0.7 means ~70% of those panels are anomalous

fraction_pos, mean_pred = calibration_curve(y_test, y_prob, n_bins=10)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(mean_pred, fraction_pos, 's-', color='blue', label='Model')
ax.plot([0,1],[0,1], 'k--', label='Perfect calibration')
ax.set_xlabel('Mean Predicted Probability')
ax.set_ylabel('Fraction of Positives')
ax.set_title('Calibration Curve\n'
             'Closer to diagonal = better calibrated risk scores')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('calibration_curve.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Brier Score: {brier:.4f}')
print('(0.0 = perfect  |  0.25 = no skill  |  lower is better)')

## 16 — Risk Score Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution by label
normal_scores  = y_prob[y_test == 0]
anomaly_scores = y_prob[y_test == 1]
axes[0].hist(normal_scores,  bins=50, alpha=0.6, color='steelblue', label='Normal')
axes[0].hist(anomaly_scores, bins=50, alpha=0.6, color='tomato',    label='Anomaly')
axes[0].set_xlabel('Risk Score P(anomaly)')
axes[0].set_ylabel('Count')
axes[0].set_title('Risk Score Distribution by True Label')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Distribution by fault type
fault_colors = {'normal':'steelblue','dusty':'orange',
                'bird_drop':'purple','cracked':'tomato'}
for ft, color in fault_colors.items():
    mask = fault_test == ft
    if mask.sum() > 0:
        axes[1].hist(y_prob[mask], bins=30, alpha=0.5,
                     color=color, label=f'{ft} (n={mask.sum()})')
axes[1].set_xlabel('Risk Score P(anomaly)')
axes[1].set_ylabel('Count')
axes[1].set_title('Risk Score Distribution by Fault Type')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('risk_score_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: risk_score_distribution.png')

## 17 — Export Trained Model

In [ ]:
# Save pipeline (scaler + model together)
# This is what deploys at the Fog layer
model_path = 'logreg_risk_model.joblib'
joblib.dump(pipeline, model_path)
print(f'Model saved: {model_path}')

# Save threshold config
import json
config = {
    'feature_cols':      FEATURE_COLS,
    'threshold_f1':      float(best_thresh.threshold),
    'threshold_recall90': float(conservative.threshold),
    'roc_auc':           float(roc_auc),
    'avg_precision':     float(avg_prec),
    'brier_score':       float(brier),
    'recommended_threshold': float(conservative.threshold),
    'notes': 'Use threshold_recall90 for RTU deployment — better to flag than miss'
}
with open('model_config.json', 'w') as f:
    json.dump(config, f, indent=2)
print(f'Config saved: model_config.json')

# Verify reload works
loaded = joblib.load(model_path)
test_pred = loaded.predict_proba(X_test[:5])[:,1]
print(f'\nReload check — first 5 risk scores: {test_pred.round(4)}')
print('Model export successful.')

## 18 — How to Use at the Fog Layer

In [ ]:
# This is exactly how the Fog node uses the model
# at inference time on incoming RTU packets

demo_code = '''
import joblib
import numpy as np

# Load once at Fog startup
pipeline = joblib.load('logreg_risk_model.joblib')
THRESHOLD = 0.35  # from model_config.json threshold_recall90

def compute_risk_score(rtu_packet):
    """
    rtu_packet: dict with telemetry + image scores
    Returns: scalar risk score in [0, 1]
    """
    feature_cols = [
        'PR', 'TEMP_DELTA', 'DC_AC_RATIO',
        'PR_ROLL_MEAN', 'PR_ROLL_STD', 'PR_SLOPE',
        'PR_DEV', 'TEMP_DELTA_SIGMA',
        'img_panel_score', 'img_dusty_score',
        'img_cracked_score', 'img_bird_drop_score'
    ]
    X = np.array([[rtu_packet[f] for f in feature_cols]])
    risk_score = pipeline.predict_proba(X)[0, 1]
    return float(risk_score)

# Example RTU packet (telemetry + YOLO image scores)
example_packet = {
    'PR': 1050.0, 'TEMP_DELTA': 14.2, 'DC_AC_RATIO': 10.21,
    'PR_ROLL_MEAN': 1100.0, 'PR_ROLL_STD': 45.0, 'PR_SLOPE': -8.5,
    'PR_DEV': 0.22, 'TEMP_DELTA_SIGMA': 0.3,
    'img_panel_score': 0.08, 'img_dusty_score': 0.79,
    'img_cracked_score': 0.07, 'img_bird_drop_score': 0.06
}

risk = compute_risk_score(example_packet)
print(f'Risk Score: {risk:.4f}')
print(f'Alert:      {"ANOMALY DETECTED" if risk >= THRESHOLD else "Normal"}')
'''

print('=== FOG LAYER INFERENCE CODE ===')
print(demo_code)

# Actually run it
example_packet = {
    'PR': 1050.0, 'TEMP_DELTA': 14.2, 'DC_AC_RATIO': 10.21,
    'PR_ROLL_MEAN': 1100.0, 'PR_ROLL_STD': 45.0, 'PR_SLOPE': -8.5,
    'PR_DEV': 0.22, 'TEMP_DELTA_SIGMA': 0.3,
    'img_panel_score': 0.08, 'img_dusty_score': 0.79,
    'img_cracked_score': 0.07, 'img_bird_drop_score': 0.06
}
X_demo = np.array([[example_packet[f] for f in FEATURE_COLS]])
risk   = pipeline.predict_proba(X_demo)[0, 1]
thresh = float(conservative.threshold)
print(f'Live test — Risk Score: {risk:.4f}')
print(f'Threshold:              {thresh:.2f}')
print(f'Decision:               {"ANOMALY DETECTED" if risk >= thresh else "Normal"}')

## 19 — Final Summary

In [ ]:
print('=' * 55)
print('LOGISTIC REGRESSION MODEL — FINAL SUMMARY')
print('=' * 55)
print(f'  ROC-AUC:              {roc_auc:.4f}')
print(f'  Average Precision:    {avg_prec:.4f}')
print(f'  Brier Score:          {brier:.4f}')
print(f'  CV AUC (5-fold):      {cv_auc.mean():.4f} ± {cv_auc.std():.4f}')
print(f'  Best F1 threshold:    {best_thresh.threshold:.2f}')
print(f'  Recall-90 threshold:  {conservative.threshold:.2f}  <- use this')
print()
print('  OUTPUT FILES')
import os
for fname, desc in [
    ('logreg_risk_model.joblib',     'Trained model for Fog deployment'),
    ('model_config.json',            'Thresholds + metadata'),
    ('roc_pr_curves.png',            'ROC and PR curves'),
    ('threshold_analysis.png',       'Threshold vs F1/Recall'),
    ('feature_importance.png',       'Feature coefficients'),
    ('calibration_curve.png',        'Probability calibration'),
    ('risk_score_distribution.png',  'Score distributions by class'),
]:
    status = 'OK' if os.path.exists(fname) else 'MISSING'
    print(f'  [{status}] {fname:35s} {desc}')
print('=' * 55)